In [1]:
%load_ext autoreload
%autoreload 2

# Creating syntax_morphology_conflicts dataset

As a result of running the script, a new database file, syntax_morphology_conflicts.db, is generated.
The database contains phrases where there is reason to suspect that the case marking of the phrase root has been incorrectly labeled.

This dataset covers **only the verbs present in the pattern database**.

Input:

- Database of transactions not covered by patterns: `uncovered_transactions.db`.

Output:

**syntax_morphology_conflicts**

| Väli              | Tüüp | Kirjeldus                                                              | Näide | Märkus          |
| ----------------- | ---- | ---------------------------------------------------------------------- | ----- | --------------- |
| id                | int  | rea unikaalne ID                                                       |       |                 |
| pattern_id        | int  | malli ID vp_data3.db baasis (patterns.pat_id)                          |       |                 |
| sentence_id       | int  | lause ID koondkorpuse lausete andmebaasis                              |       |                 |
| verb_loc          | int  | verbi asukoht lauses                                                   |       |                 |
| compound_loc      | JSON | verbi afiksaaladverbide asukohad lauses                                |       |                 |
| phrase_root_loc   | int  | fraasi juure asukoht lauses                                            |       |                 |
| verb_phrase_loc   | JSON | verbifraasi liikmete asukoht lauses (vahetud alluvad + OBL alluv case) |       |                 |
| phrase_case       | text | malli kääne                                                            |       |                 |
| phrase_deprel     | text | fraasi juure deprel                                                    |       |                 |
| verb              | text | verbi lemma                                                            |       |                 |
| verb_compound     | text | verbi afiksaaladverbid                                                 |       | eraldajaks koma |
| phrase            | text | puhastatud verbifraas                                                  |       |                 |
| phrase_root_lemma | text | fraasi juure lemma                                                     |       |                 |
| current_analysis  | text | fraasi juure analüüs andmebaasis                                       |       |                 |
| current_case      | text | fraasi juure kääne andmebaasis                                         |       |                 |
| possible_cases    | JSON | võimalikud käänded                                                     |       |                 |


In [2]:
#!pip install tqdm ipywidgets
import os
import sys
import json
import pandas as pd

from datetime import datetime
from sqlalchemy import create_engine
from sqlalchemy.orm import Session
from helpers.models import Base, SyntaxMorphologyConflict
from tqdm.notebook import tqdm
from pathlib import Path
from estnltk.taggers import VabamorfAnalyzer  # v1.7.3
from helpers.helpers import collect_misanalysed_transactions, get_cg_case

ROOT = str(Path(os.getcwd()).parent.parent.parent)
sys.path.append(f"{ROOT}/common_code")

from apriori import V33
from db_operations.db_display import display_sqlite_as_dataframe

date_time = datetime.now().strftime("%Y%m%d-%H%M%S")

## Configuration


In [3]:
DB_DIR = "./example_data"
UNCOVERED_TRANSACTIONS_DB = DB_DIR + "/uncovered_transactions.db"
SYNTAX_MORPHOLOGY_CONFILCS_TDB = DB_DIR + "/syntax_morphology_conflicts.db"

## Workflow


In [4]:
# init token analyser
morph_analyzer = VabamorfAnalyzer()

# init apriori for V33
uncovered_transactions = V33(file_path=UNCOVERED_TRANSACTIONS_DB)

In [5]:
# fetch all verbs
verbs = uncovered_transactions.execute_text("SELECT * FROM verbs").mappings().all()
display_sqlite_as_dataframe(
    db_path=UNCOVERED_TRANSACTIONS_DB,
    table_name="verbs",
    n_rows=5,
)

,verb_id,verb,verb_compound,pat_ids
0,1,aasima,,"1,2"
1,2,neelama,alla,32


In [6]:
# fetch all patterns
patterns = {
    p["pat_id"]: p
    for p in uncovered_transactions.execute_text("SELECT * FROM patterns")
    .mappings()
    .all()
}
display_sqlite_as_dataframe(
    db_path=UNCOVERED_TRANSACTIONS_DB,
    table_name="patterns",
    n_rows=5,
)

,pat_id,pattern,verb_word,verb_compound,phrase_nr,phrase_case,adp,inf_verb
0,1,aasima keda*,aasima,,1,part,,
1,2,aasima kelle kallal,aasima,,1,gen,kallal,
2,32,alla neelama mida,neelama,alla,1,part,,


### Creating dataset


In [7]:

%%time
# Create the database
engine = create_engine(f"sqlite:///{SYNTAX_MORPHOLOGY_CONFILCS_TDB}", echo=False)
Base.metadata.create_all(engine)

with Session(engine) as session:
    session.commit()

print("Database and tables created.")

# Insert data into the database in batches
batch_size = 10000
batch = []
print("Starting data collection.")
for v in tqdm(verbs):
    pat_ids = [
        int(id)
        for id in v["pat_ids"].split(",")
        if len(patterns[int(id)]["phrase_case"])
    ]
    if not pat_ids:
        print("No patterns for analysis", v)
        continue

    # Retrieve all transactions for the verb, including compound dependents
    uncovered_transactions.get_transactions(verb=v["verb"], verb_compound=v["verb_compound"], force_keep_compound=True)
    
    # More information is stored in the class's internal variable
    transactions_raw = uncovered_transactions._raw_transactions

    # Identify transactions where members had a case incompatible with the pattern
    misanalysed_transactions = collect_misanalysed_transactions(pat_ids = pat_ids, patterns=patterns, transactions_raw=transactions_raw)
   
    # rocess verb transactions by pattern
    for pat_id in misanalysed_transactions.keys():
        # Case required by the pattern
        pattern_case = patterns[pat_id]["phrase_case"]
        misanalysed_transactions2 = {}
        # If phrase members' analysis identified a case compatible with the template
        for head_id in misanalysed_transactions[pat_id]:
            for tr in transactions_raw[head_id]:
                possible_cases = [get_cg_case(analysis=a) for a in morph_analyzer.analyze_token(tr['frequent_form'])]
                if pattern_case in possible_cases:
                    tr['possible_cases'] = list(set(possible_cases))
                    misanalysed_transactions2.setdefault(head_id, []).append(tr)

        head_ids = list(misanalysed_transactions2.keys())
        
        # Skip if no compatible cases were found
        if not len(head_ids): continue

        # Retrieve verb phrase data for all head IDs at once
        phrases = [
            dict(row) for row in uncovered_transactions.get_phrases_by_head_ids(head_ids=head_ids)
        ]
        
        for i, phrase in enumerate(phrases):
            head_id = phrase['head_id']
            sentence_id = phrase['sentence_id']
            verb_loc = phrase['loc']
            verb_lemma = v['verb']
            verb_compound = v['verb_compound']
            compound_loc = [ row['loc'] for row in transactions_raw[head_id] if  row['deprel'].lower() == 'compound:prt']
            phrase_loc = sorted([verb_loc] + [row['loc'] for row in transactions_raw[head_id]])
            phrase_text = phrase['phrase']
            
            # A single phrase could contain multiple words with uncertain analysis
            # Each word is added as a separate row
            for tr in misanalysed_transactions2[head_id]:
                
                current_morph = tr['feats']
                current_case = tr['case']
                lemma_loc = tr['loc'] # Incorrectly analyzed word
                lemma_deprel = tr['deprel'].lower() # Deprel of the incorrectly analyzed word
                lemma = tr['lemma']
                possible_cases = tr['possible_cases']
                

                conflict_entry = SyntaxMorphologyConflict(
                    pattern_id=pat_id,
                    sentence_id=sentence_id,
                    verb_loc=verb_loc,
                    compound_loc=json.dumps(compound_loc) if len(compound_loc) else None,
            
                    phrase_root_loc= lemma_loc,
                    verb_phrase_loc=json.dumps(phrase_loc),
                    phrase_case=pattern_case,
                    phrase_deprel=lemma_deprel,
                    
                    verb=verb_lemma,
                    verb_compound=verb_compound,
                    phrase=phrase_text,
                    phrase_root_lemma=lemma,
                    current_analysis=current_morph,
                    current_case=current_case,
                    possible_cases=json.dumps(possible_cases)

                )
                
                batch.append(conflict_entry)

                if len(batch) >= batch_size:
                    session.bulk_save_objects(batch)
                    session.commit()
                    batch = []
                    
# Commit any remaining items in the batch
if batch:
    session.bulk_save_objects(batch)
    session.commit()

session.close()


Database and tables created.
Starting data collection.


  0%|          | 0/2 [00:00<?, ?it/s]

CPU times: user 106 ms, sys: 16.7 ms, total: 123 ms
Wall time: 133 ms


In [8]:
display_sqlite_as_dataframe(
    db_path=SYNTAX_MORPHOLOGY_CONFILCS_TDB,
    table_name="syntax_morphology_conflicts",
    n_rows=5,
)

,id,pattern_id,sentence_id,verb_loc,compound_loc,phrase_root_loc,verb_phrase_loc,phrase_case,phrase_deprel,verb,verb_compound,phrase,phrase_root_lemma,current_analysis,current_case,possible_cases
0,1,1,30442,3,null,4,"""[1, 2, 3, 4]""",part,obl,aasima,,Ma siis aasisin Deani,Dea,"prop,sg,term",term,"""[null, \""part\"", \""term\"", \""adit\"", \""nom\"", \""gen\""]"""
1,2,1,995483,5,null,3,"""[3, 4, 5]""",part,advmod,aasima,,pärast veel aasis,pärast,post,,"""[null, \""part\"", \""el\""]"""
2,3,1,1081322,8,null,9,"""[8, 9]""",part,nsubj,aasima,,aasib Taavi,Taavi,"nom,prop,sg",nom,"""[null, \""part\"", \""adit\"", \""nom\"", \""gen\""]"""
3,4,1,1156416,9,null,11,"""[9, 10, 11]""",part,obl,aasima,,aasib Leinatamm apsu üle,aps,"com,gen,sg",gen,"""[\""part\"", \""adit\"", \""gen\""]"""
4,5,1,1990874,9,null,11,"""[9, 10, 11, 24]""",part,obl,aasima,,aasisid muusikud Justamendist sai,Justamendi,"el,prop,sg",el,"""[\""part\""]"""
